# LLM Classification Finetuning — v0.3b Inference

**Model:** `google/gemma-2-2b-it` + LoRA adapter trained on DGX Spark GB10 (v0.3b 5-fold).
**Strategy:** Swap-TTA — average (A,B) + (B,A) predictions to cancel position bias.
**Offline:** base model from Kaggle Models; adapter from private dataset.

**Inputs:**
- `google/gemma-2/transformers/gemma-2-2b-it/1` — base model
- `gdataranger/gemma-2-2b-lora-v03b` — averaged LoRA adapter (or single fold for early scoring)
- `llm-classification-finetuning` — competition data

**Push workflow:**
1. `zsh scripts/push_notebook.sh v0.3b-infer`
2. Stop the auto-run immediately in Kaggle UI
3. Accelerator → GPU T4 x1 → Save Version → Save & Run All

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.44', 'peft>=0.12', 'accelerate>=0.33',
], check=True)

print('Dependencies installed')

In [ ]:
# ── Cell 2: Imports & config ───────────────────────────────────────────────────
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}: {torch.cuda.get_device_name(i)}  {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')
else:
    print('CPU only')

MAX_SEQ_LEN  = 1024
INFER_BATCH  = 16   # T4 x2 = 32 GB; 2B model ~5 GB leaves ample room
N_FOLDS      = 5
SEED         = 42
LABEL_TOKENS = ['A', 'B', 'tie']
OUTPUT       = Path('/kaggle/working')

print('Imports OK')

In [ ]:
# ── Cell 3: Path discovery ─────────────────────────────────────────────────────
# Competition data
_data_candidates = [Path('/kaggle/input/llm-classification-finetuning')]
INPUT = next((p for p in _data_candidates if (p / 'test.csv').exists()), None)
if INPUT is None:
    _found = glob.glob('/kaggle/input/**/test.csv', recursive=True)
    INPUT = Path(_found[0]).parent if _found else None
assert INPUT is not None, 'Competition data not found'
print(f'Data : {INPUT}')

# Base model — mounted from google/gemma-2/transformers/gemma-2-2b-it/1
_model_configs = [
    p for p in glob.glob('/kaggle/input/**/config.json', recursive=True)
    if 'gemma' in p.lower() and 'lora' not in p.lower() and 'adapter' not in p.lower()
]
MODEL_PATH = str(Path(_model_configs[0]).parent) if _model_configs else None
assert MODEL_PATH is not None, 'Base model not found — attach google/gemma-2/transformers/gemma-2-2b-it/1'
print(f'Model: {MODEL_PATH}')

# LoRA adapter — from gdataranger/gemma-2-2b-lora-v03b dataset
_adapter_configs = [
    p for p in glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
]
ADAPTER_PATH = str(Path(_adapter_configs[0]).parent) if _adapter_configs else None
assert ADAPTER_PATH is not None, 'Adapter not found — attach gdataranger/gemma-2-2b-lora-v03b'
print(f'Adapter: {ADAPTER_PATH}')

In [ ]:
# ── Cell 4: Load base model + adapter ─────────────────────────────────────────
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # left-pad for generation / logit readout

print('Loading base model (bf16)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map='auto',
    local_files_only=True,
    attn_implementation='flash_attention_2',
)

print('Loading LoRA adapter...')
model = PeftModel.from_pretrained(model, ADAPTER_PATH, local_files_only=True)
model.eval()
print('Model ready')

label_token_ids = [
    tokenizer.encode(t, add_special_tokens=False)[0] for t in LABEL_TOKENS
]
print(f'Label token ids: {dict(zip(LABEL_TOKENS, label_token_ids))}')

In [ ]:
# ── Cell 5: Prompt builder & inference ────────────────────────────────────────
def build_prompt(row, swap=False):
    ra = str(row['response_a'] or '')
    rb = str(row['response_b'] or '')
    if swap:
        ra, rb = rb, ra
    return (
        f'Prompt:\n{str(row["prompt"] or "")}\n\n'
        f'Response A:\n{ra}\n\n'
        f'Response B:\n{rb}\n\n'
        'Which response is better? Answer with only: A, B, or tie.'
    )

@torch.inference_mode()
def predict_proba(df, batch_size=INFER_BATCH, swap=False):
    all_probs = []
    rows = [r for _, r in df.iterrows()]
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i + batch_size]
        texts = [
            tokenizer.apply_chat_template(
                [{'role': 'user', 'content': build_prompt(row, swap=swap)}],
                tokenize=False, add_generation_prompt=True
            )
            for row in batch
        ]
        enc = tokenizer(
            texts, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_SEQ_LEN,
        ).to(model.device)
        last_logits = model(**enc).logits[:, -1, :]
        probs = torch.softmax(
            last_logits[:, label_token_ids].float(), dim=-1
        ).cpu().numpy()
        all_probs.append(probs)
        if (i // batch_size) % 100 == 0:
            print(f'  {min(i + batch_size, len(rows)):,}/{len(rows):,}')
    return np.vstack(all_probs)

print('Inference helper ready')

In [ ]:
# ── Cell 6: Load data ──────────────────────────────────────────────────────────
test = pd.read_csv(INPUT / 'test.csv')
print(f'Test: {test.shape}')

# Optional: OOF check on fold 0 validation set to verify adapter quality
train = pd.read_csv(INPUT / 'train.csv')
winner_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
train['label'] = train[winner_cols].values.argmax(axis=1).astype(int)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(train, train['label']))
_, val_idx = splits[0]
df_val = train.iloc[val_idx].reset_index(drop=True)
print(f'Fold 0 val: {len(df_val):,} rows (OOF check)')

In [ ]:
# ── Cell 7: OOF check ─────────────────────────────────────────────────────────
print('OOF inference (swap-TTA) on fold 0 val set...')
oof_fwd = predict_proba(df_val, swap=False)
oof_swp = predict_proba(df_val, swap=True)
oof_probs = (oof_fwd + oof_swp[:, [1, 0, 2]]) / 2

oof_ll = log_loss(df_val['label'].values, oof_probs)
print(f'\nFold 0 OOF log loss (swap-TTA): {oof_ll:.4f}')
print(f'v0.1 TF-IDF baseline:            1.0404')
print(f'Δ from baseline:                 {oof_ll - 1.0404:+.4f}')

In [ ]:
# ── Cell 8: Test inference & submission ───────────────────────────────────────
print('Test inference (swap-TTA)...')
test_fwd = predict_proba(test, swap=False)
test_swp = predict_proba(test, swap=True)
test_probs = (test_fwd + test_swp[:, [1, 0, 2]]) / 2

sub = pd.DataFrame({
    'id':             test['id'],
    'winner_model_a': test_probs[:, 0],
    'winner_model_b': test_probs[:, 1],
    'winner_tie':     test_probs[:, 2],
})
sub.to_csv(OUTPUT / 'submission.csv', index=False)

print(f'\nsubmission.csv: {len(sub):,} rows')
print(f'Prob sums (first 5): {test_probs.sum(axis=1)[:5].round(4)}')
print(f'NaN check: {sub.isnull().sum().sum()}')
print(sub.head())